# Store Transaction Data Analysis
**Dataset**: iamprateek/store-transaction-data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

BASE = 'abhinav/store-transaction-data'

# Load all files
files = {}
for f in sorted(os.listdir(BASE)):
    if f.endswith('.csv'):
        files[f] = pd.read_csv(os.path.join(BASE, f), low_memory=False)
        print(f'{f}: {files[f].shape[0]:,} rows x {files[f].shape[1]} cols')


## 1. Schema Overview

In [ ]:
for name, df in files.items():
    print(f"
=== {name} ===")
    print(df.dtypes)
    print(f"Nulls: {df.isnull().sum().sum()}")
    print(f"Sample:")
    display(df.head(3))


## 2. Mapping File

In [ ]:
if 'Hackathon_Mapping_File.csv' in files:
    print(files['Hackathon_Mapping_File.csv'].to_string())


## 3. Working Data Deep Dive

In [ ]:
df = files.get('Hackathon_Working_Data.csv')
if df is not None:
    print(f'Shape: {df.shape}')
    print(f'
Column types:')
    print(df.dtypes)
    print(f'
Null counts:')
    print(df.isnull().sum())
    df.describe()


In [ ]:
# Categorical distributions
if df is not None:
    cat_cols = [c for c in df.columns if df[c].dtype == 'object' or df[c].nunique() < 20]
    n = min(len(cat_cols), 8)
    if n > 0:
        fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
        axes = axes.flatten()
        for i, col in enumerate(cat_cols[:n]):
            df[col].value_counts().head(15).plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
            axes[i].set_title(f'{col}')
            axes[i].tick_params(axis='x', rotation=45)
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        plt.tight_layout()
        plt.show()


In [ ]:
# Numeric distributions
if df is not None:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    n = min(len(num_cols), 8)
    if n > 0:
        fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
        axes = axes.flatten()
        for i, col in enumerate(num_cols[:n]):
            axes[i].hist(df[col].dropna(), bins=40, edgecolor='black', alpha=0.7)
            axes[i].set_title(f'{col}')
            axes[i].axvline(df[col].mean(), color='red', linestyle='--')
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        plt.tight_layout()
        plt.show()


## 4. Correlation Analysis

In [ ]:
if df is not None:
    num_df = df.select_dtypes(include=[np.number])
    if num_df.shape[1] > 2:
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
        ax.set_title('Correlation Matrix — Working Data')
        plt.tight_layout()
        plt.show()


## 5. Ideal Data Analysis

In [ ]:
idf = files.get('Hackathon_Ideal_Data.csv')
if idf is not None:
    print(f'Shape: {idf.shape}')
    display(idf.head())
    display(idf.describe())


In [ ]:
if idf is not None:
    num_cols_i = idf.select_dtypes(include=[np.number]).columns.tolist()
    n = min(len(num_cols_i), 6)
    if n > 0:
        fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
        axes = axes.flatten()
        for i, col in enumerate(num_cols_i[:n]):
            axes[i].hist(idf[col].dropna(), bins=40, edgecolor='black', alpha=0.7, color='coral')
            axes[i].set_title(f'{col}')
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        plt.tight_layout()
        plt.show()


## 6. Relevance to Shelf Optimization / Planogram AI

**Potential Uses:**
- Store-level transaction patterns for demand forecasting
- Product category performance for shelf allocation
- Transaction volume patterns for inventory planning

**Limitations:**
- Hackathon dataset — may have synthetic/modified data
- Need to verify product-level granularity
